In [ ]:
import os

import GCRCatalogs
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from GCRCatalogs import GCRQuery
from GCRCatalogs.helpers.tract_catalogs import tract_filter

GCRCatalogs.set_root_dir("/data/scratch/dc2_nfs/")

We specify the path at which we'll save the pkl file:

In [ ]:
file_name = "dc2_lensing_catalog_objectmatch.pkl"
file_path = os.path.join("/data", "scratch", "dc2local", file_name)
file_already_populated = os.path.isfile(file_path)

---

### Object-with-truth–match

We load in the object-with-truth-match catalog and list all available quantities:

In [ ]:
object_truth_cat = GCRCatalogs.load_catalog("desc_dc2_run2.2i_dr6_object_with_truth_match")

In [ ]:
object_truth_cat.list_all_quantities()

And we fetch the variables we want:

In [ ]:
object_truth_df = object_truth_cat.get_quantities(
    quantities=[
        "cosmodc2_id_truth",
        "id_truth",
        "objectId",
        "match_objectId",
        "truth_type",
        "ra_truth",
        "dec_truth",
        "redshift_truth",
        "flux_u_truth",
        "flux_g_truth",
        "flux_r_truth",
        "flux_i_truth",
        "flux_z_truth",
        "flux_y_truth",
        "mag_u_truth",
        "mag_g_truth",
        "mag_r_truth",
        "mag_i_truth",
        "mag_z_truth",
        "mag_y_truth",
        "Ixx_pixel",
        "Iyy_pixel",
        "Ixy_pixel",
        "IxxPSF_pixel_u",
        "IxxPSF_pixel_g",
        "IxxPSF_pixel_r",
        "IxxPSF_pixel_i",
        "IxxPSF_pixel_z",
        "IxxPSF_pixel_y",
        "IyyPSF_pixel_u",
        "IyyPSF_pixel_g",
        "IyyPSF_pixel_r",
        "IyyPSF_pixel_i",
        "IyyPSF_pixel_z",
        "IyyPSF_pixel_y",
        "IxyPSF_pixel_u",
        "IxyPSF_pixel_g",
        "IxyPSF_pixel_r",
        "IxyPSF_pixel_i",
        "IxyPSF_pixel_z",
        "IxyPSF_pixel_y",
        "psf_fwhm_u",
        "psf_fwhm_g",
        "psf_fwhm_r",
        "psf_fwhm_i",
        "psf_fwhm_z",
        "psf_fwhm_y",
    ],
    native_filters=[tract_filter([3634, 3635, 3636, 3827, 3828, 3829, 3830, 4025, 4026, 4027])],
)

object_truth_df = pd.DataFrame(object_truth_df)

We create an ra/dec filter that we'll apply to CosmoDC2, as well as an equivalent (but faster) healpix filter:

In [ ]:
max_ra = np.nanmax(object_truth_df["ra_truth"])
min_ra = np.nanmin(object_truth_df["ra_truth"])
max_dec = np.nanmax(object_truth_df["dec_truth"])
min_dec = np.nanmin(object_truth_df["dec_truth"])
ra_dec_filters = [f"ra >= {min_ra}", f"ra <= {max_ra}", f"dec >= {min_dec}", f"dec <= {max_dec}"]

In [ ]:
vertices = hp.ang2vec(
    np.array([min_ra, max_ra, max_ra, min_ra]),
    np.array([min_dec, min_dec, max_dec, max_dec]),
    lonlat=True,
)
ipix = hp.query_polygon(32, vertices, inclusive=True)
healpix_filter = GCRQuery((lambda h: np.isin(h, ipix, True), "healpix_pixel"))

We see that there are around 11 million objects in the object-truth table:

In [ ]:
object_truth_df.shape

But a small portion of these objects are stars or supernovae rather than galaxies, so we filter them out:

In [ ]:
object_truth_df = object_truth_df[object_truth_df["truth_type"] == 1]

In [ ]:
object_truth_df.shape

There are also a few duplicate `cosmodc2_id`s. These arise because a small number of measured objects in `object_truth_df` are matched with the same `cosmodc2_id` in the truth table. But the only columns of `object_truth_df` that we are using are the `_truth` columns, so our merged data frame has the same truth values listed multiple times (as opposed to slighly different measured values corresponding to the multiple objects that were matched to the same `cosmodc2_id`).

In [ ]:
object_truth_df.drop_duplicates(subset=["cosmodc2_id_truth"], inplace=True)

In [ ]:
object_truth_df.shape

Here are the different ID variables in the object-truth table:

In [ ]:
object_truth_df[["cosmodc2_id_truth", "id_truth", "objectId", "match_objectId"]]

We see that `id` and `cosmodc2_id` are equivalent, at least for galaxies:

In [ ]:
(object_truth_df["cosmodc2_id_truth"] == object_truth_df["id_truth"]).mean()

And same goes for `objectId` and `match_objectId`:

In [ ]:
(object_truth_df["objectId"] == object_truth_df["match_objectId"]).mean()

Here's the distribution of magnitude in each band:

In [ ]:
_ = plt.hist(object_truth_df["mag_u_truth"], bins=50, alpha=0.3, label="u")
_ = plt.hist(object_truth_df["mag_g_truth"], bins=50, alpha=0.3, label="g")
_ = plt.hist(object_truth_df["mag_r_truth"], bins=50, alpha=0.3, label="r")
_ = plt.hist(object_truth_df["mag_i_truth"], bins=50, alpha=0.3, label="i")
_ = plt.hist(object_truth_df["mag_z_truth"], bins=50, alpha=0.3, label="z")
_ = plt.hist(object_truth_df["mag_y_truth"], bins=50, alpha=0.3, label="y")
_ = plt.legend()

And here's the distribution of redshift:

In [ ]:
_ = plt.hist(object_truth_df["redshift_truth"], bins=50)

---

### CosmoDC2

Next, we load in CosmoDC2 and list all available quantities:

In [ ]:
config_overwrite = {"catalog_root_dir": "/data/scratch/dc2_nfs/cosmoDC2_v1.1.4"}

cosmo_cat = GCRCatalogs.load_catalog("desc_cosmodc2", config_overwrite)

In [ ]:
cosmo_cat.list_all_quantities()

And we fetch the variables we want.

Some notes:
- We only pull objects in the ra/dec region from the truth table, using the filters defined above.
- ellipticity_1_true_dc2 and ellipticity_2_true_dc2 are the unlensed ellipticities that were used in the DC2 images (i.e., shear and convergence were applied to these ellipticities). ellipticity_1_true and ellipticity_2_true are the unlensed ellipticities from a subsequent bug fix described in section 2.4 of [this paper](https://arxiv.org/pdf/2101.04855). Ideally we'd want the lensed versions ellipticity_1 and ellipticity_2, which are listed [here](https://github.com/LSSTDESC/gcr-catalogs/blob/master/GCRCatalogs/SCHEMA.md) but are not available (see the above list).

In [ ]:
cosmo_df = cosmo_cat.get_quantities(
    quantities=[
        "galaxy_id",
        "ra",
        "dec",
        "ellipticity_1_true",
        "ellipticity_2_true",
        "ellipticity_1_true_dc2",
        "ellipticity_2_true_dc2",
        "shear_1",
        "shear_2",
        "convergence",
    ],
    filters=ra_dec_filters,
    native_filters=healpix_filter,
)
cosmo_df = pd.DataFrame(cosmo_df)

We see that there are around 135 million galaxies in CosmoDC2:

In [ ]:
cosmo_df.shape

Here's the ID variable in CosmoDC2. It has the same format as `cosmodc2_id_truth` in the object-with-truth-match table:

In [ ]:
cosmo_df["galaxy_id"]

---

### Merge `object_truth_df` and `cosmo_df`

Now we merge the object-with-truth-match table with CosmoDC2 using the galaxies' CosmoDC2 IDs (`galaxy_id` in `cosmo_df` and `cosmodc2_id_truth` in `object_truth_df`).

In [ ]:
merge_df = object_truth_df.merge(
    cosmo_df, left_on="cosmodc2_id_truth", right_on="galaxy_id", how="left"
)

As expected, the new data frame has the same number of rows as `object_truth_df`:

In [ ]:
merge_df.shape

In [ ]:
_ = plt.scatter(
    merge_df["ra_truth"], merge_df["dec_truth"], c=merge_df["redshift_truth"], alpha=0.005, s=1
)
_ = plt.vlines(x=[min_ra, max_ra], ymin=max_dec, ymax=min_dec)
_ = plt.hlines(y=[min_dec, max_dec], xmin=min_ra, xmax=max_ra)

We confirm that the distribution of per-band magnitude and the distribution of redshift are essentially the same as before:

In [ ]:
_ = plt.hist(merge_df["mag_u_truth"], bins=50, alpha=0.3, label="u")
_ = plt.hist(merge_df["mag_g_truth"], bins=50, alpha=0.3, label="g")
_ = plt.hist(merge_df["mag_r_truth"], bins=50, alpha=0.3, label="r")
_ = plt.hist(merge_df["mag_i_truth"], bins=50, alpha=0.3, label="i")
_ = plt.hist(merge_df["mag_z_truth"], bins=50, alpha=0.3, label="z")
_ = plt.hist(merge_df["mag_y_truth"], bins=50, alpha=0.3, label="y")
_ = plt.legend()

In [ ]:
_ = plt.hist(merge_df["redshift_truth"], bins=50)

And now we can examine shear and convergence, which were initially in `cosmo_df`:

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
_ = ax[0].hist(merge_df["shear_1"], bins=100)
_ = ax[0].set_xlabel("shear1")
_ = ax[1].hist(merge_df["shear_2"], bins=100)
_ = ax[1].set_xlabel("shear2")
_ = ax[2].scatter(merge_df["shear_1"], merge_df["shear_2"], alpha=0.05)
_ = ax[2].set_xlabel("shear1")
_ = ax[2].set_ylabel("shear2")
_ = ax[3].hist(merge_df["convergence"], bins=100)
_ = ax[3].set_xlabel("convergence")
fig.tight_layout()

We remove `ra_truth` and `dec_truth` since they're the same as `ra` and `dec`, respectively:

In [ ]:
merge_df.drop(columns=["ra_truth", "dec_truth"], inplace=True)

And finally, we remove the `_truth` suffix from the mag, flux, and redshift variables:

In [ ]:
merge_df.rename(
    columns={
        "redshift_truth": "redshift",
        "flux_u_truth": "flux_u",
        "flux_g_truth": "flux_g",
        "flux_r_truth": "flux_r",
        "flux_i_truth": "flux_i",
        "flux_z_truth": "flux_z",
        "flux_y_truth": "flux_y",
        "mag_u_truth": "mag_u",
        "mag_g_truth": "mag_g",
        "mag_r_truth": "mag_r",
        "mag_i_truth": "mag_i",
        "mag_z_truth": "mag_z",
        "mag_y_truth": "mag_y",
    },
    inplace=True,
)

---

Finally, we save `merge_df` to `file_path`:

In [ ]:
# with open(file_path, "wb") as f:
#     pkl.dump(merge_df, f)